# FinAI Agentic RAG — All 5 Agent Profiles

This notebook demonstrates every agent registered in ``fin_ai.agents.agent_library``:

| # | Agent | Specialty |
|---|-------|-----------|
| 1 | **Data_Analyst** | Quantitative financial statements analysis |
| 2 | **Market_Analyst** | Market data, sentiment, analyst consensus |
| 3 | **Research_Analyst** | Deep-dive company research (full toolkit) |
| 4 | **Thematic_Investor** | Theme-based evaluation (AI, energy transition) |
| 5 | **Research_Publisher** | Format, publish, and distribute research |

Each agent automatically discovers and registers its tools — YFinance data,
local RAG queries, model listing, and publishing — by name from the
``agent_library.py`` profile.

**Before running:** make sure ``dashboard/__init__.py`` can be imported
(setup ``VECTOR_DB_DIR``, embeddings, etc.) and local vector stores exist.

## 1. Imports & Setup

Import the agent framework, initialise the engine bridge, and configure the LLM.
The engine bridge will lazily load local FAISS vector stores on first use.

In [ ]:
import os, sys, json
from pathlib import Path

# Ensure project root is on sys.path
_project_root = Path().resolve().parent
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

# Agent framework
from fin_ai.agents.workflow import SingleAssistantRAG, SingleAssistant, register_tools
from fin_ai.agents.agent_library import library
from fin_ai.agents.engine_bridge import init_engine, query_local_rag

# Dashboard configuration + embeddings
from dashboard import VECTOR_DB_DIR, OLLAMA_BASE_URL, DEFAULT_CHAT_MODEL
from dashboard.utils import get_embeddings

# LiteLLM request
from fin_ai.core.request import ModelRequest, ModelRequestFactory, RequestPayload
from fin_ai.core.response import ModelResponse, ResponseFactory

from autogen import register_function

print("✅ Imports successful")
print(f"   Agents available: {list(library.keys())}")

ModuleNotFoundError: No module named 'fin_ai'

## 2. LLM Configuration

Configure the LLM provider.  By default uses Ollama (local).  If you have
an ``OAI_CONFIG_LIST`` file with credentials, it switches to that provider.

import autogen

# Option A: OpenAI-compatible (requires OAI_CONFIG_LIST in project root)
openai_available = Path("../OAI_CONFIG_LIST").exists()

if openai_available:
    llm_config = {
        "config_list": autogen.config_list_from_json(
            "../OAI_CONFIG_LIST",
            filter_dict={"model": ["gpt-4o", "gpt-4-0125-preview"]},
        ),
        "timeout": 120,
        "temperature": 0,
    }
    chat_provider = "openai"
else:
    # Option B: Ollama (local)
    llm_config = {
        "config_list": [{"model": DEFAULT_CHAT_MODEL, "base_url": OLLAMA_BASE_URL, "api_key": "ollama"}],
        "timeout": 120,
        "temperature": 0,
    }
    chat_provider = "ollama"

print(f"🔧 Provider: {chat_provider} | Model: {llm_config['config_list'][0].get('model', 'auto')}")

In [ ]:
## 3. Initialise the Engine Bridge

Pre-warms the RAG engine: discovers vector stores, loads their embeddings,
and builds source configs.  Dimension-mismatched stores are skipped gracefully.

NameError: name 'get_source_vector_stores' is not defined

In [ ]:
engine_status = init_engine(chat_provider=chat_provider)

print(f"✅ Engine initialised")
print(f"   Provider:    {engine_status['provider']}")
print(f"   Stores:      {engine_status['num_stores']}")
print(f"   Available:   {engine_status['available_dbs']}")
print(f"   Source cfgs: {engine_status['num_source_configs']}")

NameError: name 'available_dbs' is not defined

## 3. Helper: Build Agent LLM Config

A shared helper to create an AutoGen-compatible ``llm_config`` dict from
our notebook-level variables, plus a small retrieve_config for RAG agents.

In [ ]:
def _rag_retrieve_config(collection="notebook_demo"):
    return {
        "task": "qa",
        "vector_db": None,
        "docs_path": [],
        "chunk_token_size": 1000,
        "get_or_create": False,
        "collection_name": collection,
        "must_break_at_empty_line": False,
        "customized_prompt": (
            "Context from local FAISS stores:\n{input_context}\n\n"
            "Query: {input_question}"
        ),
    }

print("✅ Helper ready")

---

## 4. Data_Analyst — Financial Statements Analysis

The Data_Analyst can retrieve income statements, balance sheets, cash flow
statements, and stock info for any ticker.  It also has access to local RAG
for document-backed context.

In [ ]:
data_analyst = SingleAssistantRAG(
    "Data_Analyst",
    llm_config,
    human_input_mode="NEVER",
    max_consecutive_auto_reply=8,
    code_execution_config=False,
    retrieve_config=_rag_retrieve_config("data_analyst_demo"),
)

data_analyst.chat(
    "Get the latest financial data for AAPL. "
    "Retrieve the income statement and balance sheet. "
    "Summarise: revenue, net income, total assets, and debt levels."
)

print("\n✅ Data_Analyst demo complete.")

---

## 5. Market_Analyst — Market Data & Sentiment

The Market_Analyst fetches live prices, analyst recommendations, and stock
info.  It can also check which LLM providers are available.

In [ ]:
market_analyst = SingleAssistantRAG(
    "Market_Analyst",
    llm_config,
    human_input_mode="NEVER",
    max_consecutive_auto_reply=8,
    code_execution_config=False,
    retrieve_config=_rag_retrieve_config("market_analyst_demo"),
)

market_analyst.chat(
    "Get the latest stock price info and analyst recommendations for NVDA. "
    "Also check which LLM providers are available. "
    "Provide a concise market summary."
)

print("\n✅ Market_Analyst demo complete.")

---

## 6. Research_Analyst — Deep-Dive Research

The Research_Analyst has the full toolkit: all YFinance tools, local RAG,
citations, vector store listing, and model listing.  It cross-references
live data with document context.

In [ ]:
research_analyst = SingleAssistantRAG(
    "Research_Analyst",
    llm_config,
    human_input_mode="NEVER",
    max_consecutive_auto_reply=10,
    code_execution_config=False,
    retrieve_config=_rag_retrieve_config("research_demo"),
)

research_analyst.chat(
    "Research MSFT thoroughly. "
    "Get the income statement, balance sheet, cash flow, stock info, "
    "and analyst recommendations. "
    "Also list available vector stores. "
    "Produce a structured research brief covering: "
    "1) Financial health 2) Valuation 3) Risks 4) Overall assessment."
)

print("\n✅ Research_Analyst demo complete.")

---

## 7. Thematic_Investor — Theme-Based Evaluation

The Thematic_Investor evaluates companies through long-term structural themes
(e.g. AI, energy transition).  It uses financial data to quantify thematic
exposure and competitive positioning.

In [ ]:
thematic_investor = SingleAssistantRAG(
    "Thematic_Investor",
    llm_config,
    human_input_mode="NEVER",
    max_consecutive_auto_reply=8,
    code_execution_config=False,
    retrieve_config=_rag_retrieve_config("thematic_demo"),
)

thematic_investor.chat(
    "Evaluate NVDA through the lens of the AI theme. "
    "Get financial data (income statement, balance sheet, stock info, "
    "analyst recommendations). "
    "Then assess: 1) How much of NVDA's revenue is AI-driven? "
    "2) Competitive position in AI chips 3) Thematic growth outlook "
    "4) Produce a thematic scorecard."
)

print("\n✅ Thematic_Investor demo complete.")

---

## 8. Research_Publisher — Format & Publish Research

The Research_Publisher takes research content and produces polished HTML or
PDF reports, saved to ``published_research/``.  If SMTP is configured, it
can also email the report.

This agent uses ``SingleAssistant`` (not RAG) because it works with content
passed directly in the prompt.  It has access to publishing tools and can
also query RAG for supplemental data.

In [ ]:
from fin_ai.agents.engine_bridge import publish_research_report

publisher = SingleAssistant(
    "Research_Publisher",
    llm_config,
    human_input_mode="NEVER",
    max_consecutive_auto_reply=8,
    code_execution_config=False,
)

publisher.chat(
    "Publish the following research report as HTML. "
    "Use 'NVDA AI Analysis' as the title.\n\n"
    "# NVDA AI Analysis\n\n"
    "## Financial Overview\n"
    "NVIDIA (NVDA) is the dominant player in AI chips with ~80% market share.\n\n"
    "## Key Metrics\n"
    "- Revenue (TTM): $60B\n"
    "- Net Income: $20B\n"
    "- P/E Ratio: 65x\n"
    "- Market Cap: $1.8T\n\n"
    "## AI Exposure\n"
    "Data Center revenue (AI-driven): ~80% of total revenue.\n\n"
    "## Competitive Position\n"
    "Strong moat via CUDA ecosystem.  Competition from AMD (MI300) and "
    "custom ASICs (Google TPU, Amazon Trainium).\n\n"
    "## Verdict\n"
    "**Bullish** — AI investment cycle still early.  Valuation is a concern "
    "but justified by growth."
)

print("\n✅ Research_Publisher demo complete.")
print("Check the 'published_research/' directory for the generated report.")

---

## 9. Agent Pipeline — Multi-Agent Workflow

Chain multiple agents together with dependencies using ``AgentPipeline``.
This example runs Research_Analyst → Research_Publisher to analyse a company
and automatically publish the report.

In [ ]:
from fin_ai.agents.scheduler import AgentTask, AgentPipeline

pipeline = AgentPipeline(
    name="NVDA_Research_To_Publish",
    llm_config=llm_config,
)

# Task 1: Research_Analyst gathers data
pipeline.add_task(AgentTask(
    name="research_nvda",
    prompt=(
        "Research NVDA thoroughly.  Get financial data and analyst recs. "
        "Produce a structured research brief."
    ),
    agent_config="Research_Analyst",
    metadata={"retrieve_config": _rag_retrieve_config("pipeline_research")},
))

# Task 2: Research_Publisher formats and publishes the research
pipeline.add_task(AgentTask(
    name="publish_nvda",
    prompt=(
        "Publish the NVDA research as an HTML report titled 'NVDA Research Brief'. "
        "Use the analysis from the previous step."
    ),
    agent_config="Research_Publisher",
    depends_on=["research_nvda"],
))

print("✅ Pipeline defined")
for t in pipeline._tasks:
    deps = f" → depends on: {', '.join(t.depends_on)}" if t.depends_on else ""
    print(f"  • {t.name} [{t.agent_config}]{deps}")

---

## 10. Quick Reference: All Agents & Their Tools

| Agent | Tools |
|-------|-------|
| **Data_Analyst** | `get_financial_snapshot`, `get_income_stmt`, `get_balance_sheet`, `get_cash_flow`, `get_stock_dividends`, `query_local_rag`, `query_with_routed_rag`, `get_source_citations`, `list_vector_stores`, `get_provider_info` |
| **Market_Analyst** | `get_stock_data`, `get_stock_info`, `get_company_info`, `get_analyst_recommendations`, `query_local_rag`, `query_with_routed_rag`, `get_financial_snapshot`, `list_available_models`, `get_provider_info` |
| **Research_Analyst** | **All YFinance tools** + all RAG tools + model listing + citations |
| **Thematic_Investor** | `get_stock_info`, `get_company_info`, `get_income_stmt`, `get_balance_sheet`, `get_analyst_recommendations`, `get_financial_snapshot`, RAG tools |
| **Research_Publisher** | `publish_research_report`, `publish_research_html`, `publish_research_pdf`, `send_research_email`, RAG tools |